In [2]:
import torch
import torch.nn as nn
import pandas as pd

In [3]:
df = pd.read_csv('dataset/fra.csv')

In [4]:
import spacy
from spacy.tokens import DocBin

# Load models (disable components you don't need to save HUGE amounts of time)
nlp_eng = spacy.load("en_core_web_sm", disable=["ner", "parser"])
nlp_fr = spacy.load("fr_core_news_sm", disable=["ner", "parser"])

def process_and_save(column_data, nlp_model, output_path):
    doc_bin = DocBin()
    
    # Use n_process to parallelize (use 2 or 4 depending on your CPU)
    # batch_size=2000 is usually a sweet spot for large datasets
    for doc in nlp_model.pipe(column_data, n_process=2, batch_size=2000):
        doc_bin.add(doc)
        
        # Periodically save or just save at the end if you have enough RAM
        # For millions of rows, you might want to save in chunks of 100k
    
    doc_bin.to_disk(output_path)
    print(f"Saved to {output_path}")

# Process English
process_and_save(df['english'].astype(str), nlp_eng, "eng_data.spacy")

# Process French
process_and_save(df['french'].astype(str), nlp_fr, "fr_data.spacy")

Saved to eng_data.spacy
Saved to fr_data.spacy


In [17]:
from collections import Counter
word_freq = Counter()
doc_bin = DocBin().from_disk("eng_data.spacy")
# Stream through the docs to save RAM
for doc in doc_bin.get_docs(nlp_eng.vocab):
    for token in doc:
        if not token.is_punct and not token.is_space:
            word_freq[token.text.lower()] += 1

min_freq = 1
vocab = [word for word, count in word_freq.items() if count >= min_freq]

# Add special tokens
# 0 is usually reserved for Padding, 1 for Unknown words
special_tokens = ["<PAD>", "<UNK>", "<SOS>", "<EOS>"]
word2index_eng = {word: i for i, word in enumerate(special_tokens + vocab)}
index2word_eng = {i: word for i, word in enumerate(special_tokens + vocab)}

print(f"Total Vocabulary Size: {len(word2index_eng)}")

Total Vocabulary Size: 16720


In [18]:
doc_bin = DocBin().from_disk("fr_data.spacy")
# Stream through the docs to save RAM
for doc in doc_bin.get_docs(nlp_fr.vocab):
    for token in doc:
        if not token.is_punct and not token.is_space:
            word_freq[token.text.lower()] += 1

min_freq = 1
vocab = [word for word, count in word_freq.items() if count >= min_freq]

# Add special tokens
# 0 is usually reserved for Padding, 1 for Unknown words
special_tokens = ["<PAD>", "<UNK>", "<SOS>", "<EOS>"]
word2index_fr = {word: i for i, word in enumerate(special_tokens + vocab)}
index2word_fr = {i: word for i, word in enumerate(special_tokens + vocab)}

print(f"Total Vocabulary Size: {len(word2index_fr)}")

Total Vocabulary Size: 42455


In [82]:
class CBOWDataset(torch.utils.data.Dataset):
    def __init__(self, nlp, word2index, root):
        self.data = []
        doc_bin = DocBin().from_disk(root)
        for doc in doc_bin.get_docs(nlp.vocab):
            sentence = []
            for token in doc:
                if not token.is_punct and not token.is_space:
                    sentence.append(word2index[token.text.lower()])
                if len(sentence) >= 5:
                    for i in range(2, len(sentence)-2):
                        context = [sentence[i-2], sentence[i-1], sentence[i+1], sentence[i+2]]
                        label = sentence[i]
                        self.data.append((torch.tensor(context), torch.tensor(label)))
    def __getitem__(self, index):
        return self.data[index]
    def __len__(self):
        return len(self.data)

In [83]:
dataset_eng = CBOWDataset(nlp_eng, word2index_eng, "eng_data.spacy")
dataloader_eng = torch.utils.data.DataLoader(dataset_eng, 64, True)
dataset_fr = CBOWDataset(nlp_fr, word2index_fr, "fr_data.spacy")
dataloader_fr = torch.utils.data.DataLoader(dataset_fr, 64, True)

In [84]:
class Word2Vec(nn.Module):
    def __init__(self, dimensionality, vocab_size):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, dimensionality)
        self.linear = nn.Linear(dimensionality*4, vocab_size)
    def forward(self, x):
        embeds = self.embeddings(x)
        embeds = embeds.flatten(start_dim=1)
        out = self.linear(embeds)
        return out

In [85]:
Eng_model = Word2Vec(50, len(word2index_eng))
Fr_model = Word2Vec(50, len(word2index_fr))

In [105]:
from tqdm import tqdm
loss_fn = nn.CrossEntropyLoss()
LR = 0.001
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

Eng_model = Eng_model.to(DEVICE)
Fr_model = Fr_model.to(DEVICE)
def trainer(model, dataloader, EPOCHS = 20):
    optimizer = torch.optim.Adam(model.parameters(), LR)
    model.train()
    for epoch in range(EPOCHS):
        net_loss = 0
        net_acc = 0
        loop = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
        for context, word in loop:
            context = context.to(DEVICE)
            word = word.to(DEVICE)
            preds = model(context)
            loss = loss_fn(preds, word)
            net_loss += loss.item()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            pred_labels = torch.argmax(preds, 1)
            acc = (pred_labels == word).float().mean().item() * 100
            net_acc += acc
        net_loss /= len(dataloader)
        net_acc /= len(dataloader)

        print(f"Loss = {net_loss:.5f}")
        print(f"Accuracy = {net_acc:.3f}%")

In [94]:
trainer(Eng_model, dataloader_eng)

Loss = 1.90476
Accuracy = 59.405%


Loss = 1.56682
Accuracy = 63.732%


Loss = 1.45190
Accuracy = 65.680%


Loss = 1.38228
Accuracy = 66.988%


Loss = 1.33399
Accuracy = 67.863%


...

Loss = 1.15422
Accuracy = 71.221%


Loss = 1.14820
Accuracy = 71.352%


Loss = 1.14315
Accuracy = 71.441%


Loss = 1.13757
Accuracy = 71.546%


Loss = 1.13360
Accuracy = 71.645%


In [96]:
torch.save(Eng_model.embeddings.weight, 'eng_vectors.pt')

In [106]:
trainer(Fr_model, dataloader_fr, 10)
torch.save(Fr_model.embeddings.weight, 'fr_vectors.pt')

Loss = 2.20150
Accuracy = 56.481%


Loss = 1.59658
Accuracy = 65.449%


Loss = 1.38192
Accuracy = 68.949%


Loss = 1.26471
Accuracy = 70.949%


Loss = 1.19810
Accuracy = 72.166%


Loss = 1.15416
Accuracy = 72.952%


Loss = 1.12255
Accuracy = 73.550%


Loss = 1.09894
Accuracy = 73.977%


Loss = 1.08118
Accuracy = 74.284%


Loss = 1.06567
Accuracy = 74.582%


In [97]:
import pickle
with open('word2index_eng.pkl', 'wb') as pickle_file:
    pickle.dump(word2index_eng, pickle_file, protocol=pickle.HIGHEST_PROTOCOL)
with open('word2index_fr.pkl', 'wb') as pickle_file:
    pickle.dump(word2index_fr, pickle_file, protocol=pickle.HIGHEST_PROTOCOL)

In [107]:
Fr_model.embeddings.weight

Parameter containing:
tensor([[ 0.0528,  1.3756, -0.7318,  ..., -0.1577,  0.6369, -1.1419],
        [ 0.9006,  2.4718, -0.2427,  ..., -1.4633, -1.2336,  1.2460],
        [ 1.0796, -0.3591,  1.7865,  ..., -0.5044,  0.5571, -0.8825],
        ...,
        [-0.6588, -1.0737,  2.7167,  ..., -2.9470, -1.4516, -0.8692],
        [ 0.9817, -0.3874,  2.5900,  ...,  2.5929,  1.5688, -5.7416],
        [-3.3572,  2.9530,  2.7494,  ..., -2.7786, -3.1275,  0.4380]],
       device='mps:0', requires_grad=True)